# 69 Face-detectability check

Render the anonymized mesh from a sweep of viewpoints and run the MediaPipe Face Detector on every view. For vertex deletion the expected outcome is zero detections: there is no face surface left for the detector to latch onto. Detection counts are reported for every subject (Table 4.7); the contact-sheet figure is rendered for Subject 17 only, because thesis data-sharing rules allow identifiable renders for Subject 17 only.

Populates Figure 4.6 (contact sheet) and Table 4.7 (per-subject detection counts) of the thesis.

Dependency: `mediapipe`. Install into the cedalion env with `pip install mediapipe` if not yet present.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_helpers import (
    SUBJECTS, subject_paths, load_raw, load_anon, load_landmarks,
    available_subjects, missing_report, run_pipeline,
)

import numpy as np
import pandas as pd

OUT_DIR = pathlib.Path('thesis_results_out')
OUT_DIR.mkdir(exist_ok=True)

import pyvista as pv

pv.OFF_SCREEN = True

import cv2  # for BGR -> RGB conversion



# Yaw/pitch sweep in degrees

YAWS = list(range(-90, 91, 30))   # -90 .. 90 step 30

PITCHES = [-20, 0, 20]

WINDOW = (640, 640)

GREY = (200, 200, 200)

## 1. Render sweep

Rotate the camera around the anonymized head in yaw/pitch, save each frame to disk so MediaPipe can read it.

In [ ]:
from cedalion.vtktutils import trimesh_to_vtk_polydata



def render_sweep(surface, out_dir, subject_n):

    out_dir.mkdir(parents=True, exist_ok=True)

    poly = pv.wrap(trimesh_to_vtk_polydata(surface.mesh))

    files = []

    for yaw in YAWS:

        for pitch in PITCHES:

            pvplt = pv.Plotter(off_screen=True, window_size=WINDOW)

            pvplt.add_mesh(poly, color=[c/255 for c in GREY], smooth_shading=True)

            pvplt.set_background('white')

            pvplt.enable_anti_aliasing('ssaa')

            pvplt.view_xz()

            pvplt.camera.azimuth = 180 + yaw

            pvplt.camera.elevation = pitch

            pvplt.camera.zoom(1.3)

            fn = out_dir / f'subject{subject_n}_yaw{yaw:+04d}_pitch{pitch:+03d}.png'

            pvplt.screenshot(str(fn))

            pvplt.close()

            files.append((yaw, pitch, fn))

    return files

## 2. MediaPipe detector

Uses the short-range Face Detection model. Confidence threshold is left at the default 0.5; we also record the maximum confidence observed per view for the table.

In [ ]:
import mediapipe as mp

mp_face_detection = mp.solutions.face_detection



def detect_faces(image_path):

    img = cv2.imread(str(image_path))

    if img is None:

        return 0, 0.0

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    with mp_face_detection.FaceDetection(

        model_selection=0, min_detection_confidence=0.5

    ) as fd:

        result = fd.process(rgb)

    if result.detections is None:

        return 0, 0.0

    confidences = [d.score[0] for d in result.detections]

    return len(confidences), float(max(confidences))

## 3. Per-subject sweep + detection

In [ ]:
view_root = OUT_DIR / 'detectability_views'

rows = []

for n in SUBJECTS:

    if not subject_paths(n).anon_exists:

        print(f'skipping Subject{n}: no anon .obj')

        continue

    print(f'--- Subject{n} ---')

    surface = load_anon(n)

    files = render_sweep(surface, view_root / f'subject{n}', n)

    hits = 0

    max_conf = 0.0

    n_views = len(files)

    for yaw, pitch, fn in files:

        k, c = detect_faces(fn)

        hits += int(k > 0)

        if c > max_conf:

            max_conf = c

    rows.append({

        'subject': n,

        'n_views': n_views,

        'detector_hits': hits,

        'max_confidence': max_conf,

    })

    print(rows[-1])

## 4. Summary table

In [ ]:
df = pd.DataFrame(rows).sort_values('subject').reset_index(drop=True)

df

## 5. Save CSV

In [ ]:
out_csv = OUT_DIR / 'detectability_summary.csv'

df.to_csv(out_csv, index=False)

print(f'Wrote {out_csv}')